In [79]:
import pandas as pd
import numpy as np
import os
import utils as ut

In [80]:
path_logits = "F:\TFG\code\logs\\"
raw_data = "F:\TFG\datasets/raw_datasets/datalake.csv"

In [81]:
# FIELDS

ownRenames = {
                    "predictions" : "prediction",
                    'matchId' : 'match',
                    "Team_H": "HomeTeam",
                    "Team_A": "AwayTeam",
                    "prob_draw": "draw",
                    "prob_home": "home",
                    "prob_away": "away"
            }

### SEQUENTIAL OUTPUTS

In [153]:
read = lambda exp,date: ut.read_data(path_logits + exp + ".csv") if date else pd.read_csv(path_logits + exp + ".csv",sep=';',decimal=',',)
experiment_own = read("exp_comp_bal_fac_ss_1_outputs",True).query(f'split=="Test" and version==47').rename(columns=ownRenames).groupby("match").first().reset_index()
experiment_top1 = read("piRating_sequential",False).groupby("match").first().reset_index()
experiment_tabnet = read("tabnet_sequential_logits",True).groupby("match").first().reset_index()

f:\TFG\code\experiments\utils.py:20: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path,sep=';',decimal=',',parse_dates=['Date'],date_format="%d/%m/%Y")


In [183]:
experiment_own

,match,Div,Date,season,idTeam_H,idTeam_A,HomeTeam,AwayTeam,FTG_H,FTG_A,label,split,epoch,draw,home,away,prediction,version
0,4912,D1,2017-01-20,T16-17,15,2,Freiburg,Bayern Munich,1,2,2,Test,23500.0,0.291488,0.100018,0.608494,2.0,47.0
1,4921,D1,2017-01-27,T16-17,33,12,Schalke 04,Ein Frankfurt,0,1,2,Test,23500.0,0.355917,0.403696,0.240387,1.0,47.0
2,4922,D1,2017-01-28,T16-17,7,13,Darmstadt,FC Koln,1,6,2,Test,23500.0,0.367570,0.215314,0.417116,2.0,47.0
3,4925,D1,2017-01-28,T16-17,32,21,RB Leipzig,Hoffenheim,2,1,1,Test,23500.0,0.294386,0.566240,0.139374,1.0,47.0
4,4926,D1,2017-01-28,T16-17,41,2,Werder Bremen,Bayern Munich,1,2,2,Test,23500.0,0.283442,0.090201,0.626357,2.0,47.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5841,56764,SP2,2021-02-05,T20-21,249,277,Castellon,Logrones,0,0,0,Test,23500.0,0.374562,0.336016,0.289423,0.0,47.0
5842,56765,SP2,2021-02-05,T20-21,272,305,Leganes,Sp Gijon,0,0,0,Test,23500.0,0.360584,0.373002,0.266414,1.0,47.0
5843,56766,SP2,2021-02-05,T20-21,271,292,Las Palmas,Ponferradina,2,0,1,Test,23500.0,0.365648,0.375281,0.259070,1.0,47.0
5844,56767,SP2,2021-03-05,T20-21,299,312,Sabadell,Vallecano,2,0,1,Test,23500.0,0.345407,0.181980,0.472613,2.0,47.0


In [184]:
experiment_own.query("HomeTeam=='Las Palmas' and AwayTeam=='Ponferradina' and label==1") 

,match,Div,Date,season,idTeam_H,idTeam_A,HomeTeam,AwayTeam,FTG_H,FTG_A,label,split,epoch,draw,home,away,prediction,version
5843,56766,SP2,2021-02-05,T20-21,271,292,Las Palmas,Ponferradina,2,0,1,Test,23500.0,0.365648,0.375281,0.25907,1.0,47.0


In [185]:
experiment_tabnet.query("HomeTeam=='Las Palmas' and AwayTeam=='Ponferradina' and label==1")

,match,Date,Div,HomeTeam,AwayTeam,draw,home,away,prediction,label
5082,56313,2020-05-07,SP2,Las Palmas,Ponferradina,0.226656,0.637572,0.135771,1,1
5535,56766,2021-02-05,SP2,Las Palmas,Ponferradina,0.226671,0.641190,0.132139,1,1


In [187]:
experiment_top1#.query("match==56766")

,match,draw,home,away,prediction,label
0,5,0.206433,0.668897,0.124670,1,1
1,35,0.454135,0.352056,0.193809,0,0
2,39,0.288267,0.426748,0.284985,1,1
3,45,0.291792,0.620736,0.087472,1,1
4,49,0.315260,0.427324,0.257415,1,0
...,...,...,...,...,...,...
8262,58525,0.384882,0.411060,0.204057,1,1
8263,58529,0.352697,0.398782,0.248521,1,1
8264,58535,0.261142,0.567920,0.170938,1,2
8265,58539,0.316697,0.250706,0.432597,2,0


In [154]:
np.bincount(experiment_own.label), np.bincount(experiment_top1.label), np.bincount(experiment_tabnet.label)

(array([1506, 2596, 1744], dtype=int64),
 array([2276, 3804, 2187], dtype=int64),
 array([1455, 2373, 1721], dtype=int64))

In [155]:
np.bincount(experiment_own.prediction), np.bincount(experiment_top1.prediction), np.bincount(experiment_tabnet.prediction)

(array([1797, 2377, 1672], dtype=int64),
 array([ 588, 5324, 2355], dtype=int64),
 array([   0, 4515, 1034], dtype=int64))

In [156]:
print("TOTAL ACCURACY:")
(experiment_own.prediction==experiment_own.label).mean(), (experiment_top1.prediction==experiment_top1.label).mean(), (experiment_tabnet.prediction==experiment_tabnet.label).mean()

TOTAL ACCURACY:


(0.48717071501881626, 0.5523164388532721, 0.48441160569471975)

In [173]:
# cols = ['match','Div','Date','season','HomeTeam','AwayTeam','label','prediction','draw','home','away']
cols = ['match','prediction','draw','home','away']
experiments = (experiment_own[cols].merge(experiment_top1[cols],on='match',how='inner',suffixes=('_own','_top'))
                     .merge(experiment_tabnet.rename(columns={col:col+"_tab" for col in cols[1:]}),on='match',how='inner')
 ).sort_values("Date").set_index('match')

experiments = experiments[[
        'Date', 'Div', 'HomeTeam', 'AwayTeam','label', 'draw_own', 'home_own','away_own', 'prediction_own', 
        'draw_top','home_top','away_top', 'prediction_top','draw_tab','home_tab', 'away_tab', 'prediction_tab'
                        ]]

experiments

,Date,Div,HomeTeam,AwayTeam,label,draw_own,home_own,away_own,prediction_own,draw_top,home_top,away_top,prediction_top,draw_tab,home_tab,away_tab,prediction_tab
match,,,,,,,,,,,,,,,,,
25669,2019-01-02,F1,Lille,Nice,1,0.368642,0.326995,0.304363,0.0,0.225211,0.684873,0.089916,1,0.209295,0.691797,0.098908,1
43959,2019-01-02,SP1,Huesca,Valladolid,1,0.372546,0.264583,0.362870,0.0,0.373931,0.214557,0.411512,2,0.290957,0.373526,0.335516,1
43999,2019-01-03,SP1,Vallecano,Girona,2,0.376225,0.282114,0.341662,0.0,0.265991,0.202418,0.531591,2,0.305259,0.276844,0.417898,2
15485,2019-01-04,E0,Arsenal,Newcastle,1,0.312817,0.525233,0.161951,1.0,0.085434,0.895772,0.018794,1,0.247379,0.622931,0.129690,1
55789,2019-01-04,SP2,Zaragoza,Gimnastic,1,0.345340,0.446425,0.208235,1.0,0.335482,0.562125,0.102394,1,0.247612,0.589494,0.162894,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26279,2021-06-01,F1,Brest,Nice,1,0.375277,0.292833,0.331890,0.0,0.304181,0.474050,0.221769,1,0.300982,0.487102,0.211916,1
26285,2021-06-01,F1,Lyon,Lens,1,0.315865,0.514408,0.169727,1.0,0.179362,0.720824,0.099814,1,0.181844,0.713549,0.104607,1
44715,2021-06-02,SP1,Levante,Granada,0,0.374845,0.346427,0.278727,0.0,0.329475,0.367883,0.302642,1,0.304842,0.439152,0.256006,1


In [167]:
predictions = experiments[["label"]+[col for col in experiments.columns if col.startswith("prediction")]]
predictions

,label,prediction_own,prediction_top,prediction_tab
match,,,,
25669,1,0.0,1,1
43959,1,0.0,2,1
43999,2,0.0,2,2
15485,1,1.0,1,1
55789,1,1.0,1,1
...,...,...,...,...
26279,1,0.0,1,1
26285,1,1.0,1,1
44715,0,0.0,1,1


In [168]:
np.bincount(predictions.label)

array([161, 257, 173], dtype=int64)

In [169]:
np.bincount(predictions.prediction_own), np.bincount(predictions.prediction_top), np.bincount(predictions.prediction_tab)

(array([187, 245, 159], dtype=int64),
 array([ 31, 404, 156], dtype=int64),
 array([  0, 474, 117], dtype=int64))

In [170]:
(predictions.label==predictions.prediction_own).mean(), (predictions.label==predictions.prediction_top).mean(), (predictions.label==predictions.prediction_tab).mean(), 

(0.4686971235194585, 0.5532994923857868, 0.4754653130287648)

In [175]:
ut.save_dataframe(experiments,path_logits,"experiments_sequential")

In [176]:
experiments.dtypes

Date               object
Div                object
HomeTeam           object
AwayTeam           object
label               int64
draw_own          float64
home_own          float64
away_own          float64
prediction_own    float64
draw_top          float64
home_top          float64
away_top          float64
prediction_top      int64
draw_tab          float64
home_tab          float64
away_tab          float64
prediction_tab      int64
dtype: object

#### end